# Conversos EXCEL a CSV para Corfo

In [103]:
#Librerias Necesarias
import pandas as pd
from pandas.api.types import is_numeric_dtype
import numpy as np
from typing import Iterable, Optional
from __future__ import annotations
import re
import math
import unicodedata

import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")


## Nombre del archivo

In [104]:
# Funcion para cargar la extension .xlsx de excel
def cargar_archivo(nombre: str) -> str:
    nombre = nombre.strip()
    if not nombre:
        raise ValueError("Debes escribir un nombre de archivo.")
    
    return nombre if nombre.lower().endswith(".xlsx") else f"{nombre}.xlsx"


### Poner el nombre del archivo y código del proyecto entre las comillas ""

In [105]:
PI_Proyecto = "PI-6592"

In [106]:
ArchivoNombre = cargar_archivo("ArchivoPrueba")

### Leer el mensaje de abajo

In [107]:
### Cargar las hojas
HOJA_RRHH_OK = False
dfRRHH = None

try:
    # Usa engine='openpyxl' para .xlsx
    dfRRHH = pd.read_excel(ArchivoNombre, sheet_name='RRHH', engine='openpyxl')
    HOJA_RRHH_OK = True
except ValueError as e:
    # Cuando la hoja no existe, pandas típicamente levanta ValueError
    print("La hoja 'RRHH' no existe en el archivo. Se omitirán módulos dependientes.")
    HOJA_RRHH_OK = False
except FileNotFoundError:
    print(f"El archivo '{ArchivoNombre}' no se encontró.")
    HOJA_RRHH_OK = False
except Exception as e:
    # Otros errores no esperados: puedes decidir si detener o continuar
    print(f"Ocurrió un error al leer el Excel: {e}")
    print("\033[33m ⚠️Es posible que tenga el archivo abierto mientras se intenta leer, cierre el archivo para poder leerlo \033[0m")
    HOJA_RRHH_OK = False
###--------------------------------------------------------    
### Gastos de Operación
HOJA_GOperacional_OK = False
dfOperaciones = None
try:
    # Usa engine='openpyxl' para .xlsx
    dfOperaciones = pd.read_excel(ArchivoNombre, sheet_name='Gastos de Operación', engine='openpyxl')
    HOJA_GOperacional_OK  = True
except ValueError as e:
    # Cuando la hoja no existe, pandas típicamente levanta ValueError
    print("La hoja 'Gastos de Operación' no existe en el archivo. Se omitirán módulos dependientes.")
    HOJA_GOperacional_OK  = False
except FileNotFoundError:
    print(f"El archivo '{ArchivoNombre}' no se encontró.")
    HOJA_GOperacional_OK  = False
except Exception as e:
    # Otros errores no esperados: puedes decidir si detener o continuar
    print(f"Ocurrió un error al leer el Excel: {e}")
    print("\033[33m ⚠️Es posible que tenga el archivo abierto mientras se intenta leer, cierre el archivo para poder leerlo \033[0m")
    HOJA_GOperacional_OK  = False

###--------------------------------------------------------    
### Pers.Juridica
HOJA_PersJuridica_OK = False
dfPersJuridica = None
try:
    # Usa engine='openpyxl' para .xlsx
    dfPersJuridica = pd.read_excel(ArchivoNombre, sheet_name='Pers.Juridica', engine='openpyxl')
    HOJA_PersJuridica_OK  = True
except ValueError as e:
    # Cuando la hoja no existe, pandas típicamente levanta ValueError
    print("La hoja 'Pers.Juridica' no existe en el archivo. Se omitirán módulos dependientes.")
    HOJA_PersJuridica_OK  = False
except FileNotFoundError:
    print(f"El archivo '{ArchivoNombre}' no se encontró.")
    HOJA_PersJuridica_OK  = False
except Exception as e:
    # Otros errores no esperados: puedes decidir si detener o continuar
    print(f"Ocurrió un error al leer el Excel: {e}")
    print("\033[33m ⚠️Es posible que tenga el archivo abierto mientras se intenta leer, cierre el archivo para poder leerlo \033[0m")
    HOJA_PersJuridica_OK  = False


###--------------------------------------------------------    
### Arriendo
HOJA_Arriendo_OK = False
dfArriendo = None
try:
    # Usa engine='openpyxl' para .xlsx
    dfArriendo = pd.read_excel(ArchivoNombre, sheet_name='Arriendo', engine='openpyxl')
    HOJA_Arriendo_OK  = True
except ValueError as e:
    # Cuando la hoja no existe, pandas típicamente levanta ValueError
    print("La hoja 'Arriendo' no existe en el archivo. Se omitirán módulos dependientes.")
    HOJA_Arriendo_OK  = False
except FileNotFoundError:
    print(f"El archivo '{ArchivoNombre}' no se encontró.")
    HOJA_Arriendo_OK  = False
except Exception as e:
    # Otros errores no esperados: puedes decidir si detener o continuar
    print(f"Ocurrió un error al leer el Excel: {e}")
    print("\033[33m ⚠️Es posible que tenga el archivo abierto mientras se intenta leer, cierre el archivo para poder leerlo \033[0m")
    HOJA_Arriendo_OK = False


###--------------------------------------------------------    
### Servicios Básicos
HOJA_SSBB_OK = False
dfSSBB = None
try:
    # Usa engine='openpyxl' para .xlsx
    dfSSBB = pd.read_excel(ArchivoNombre, sheet_name='Servicios Básicos', engine='openpyxl')
    HOJA_SSBB_OK  = True
except ValueError as e:
    # Cuando la hoja no existe, pandas típicamente levanta ValueError
    print("La hoja 'Servicios Básicos' no existe en el archivo. Se omitirán módulos dependientes.")
    HOJA_SSBB_OK  = False
except FileNotFoundError:
    print(f"El archivo '{ArchivoNombre}' no se encontró.")
    HOJA_SSBB_OK  = False
except Exception as e:
    # Otros errores no esperados: puedes decidir si detener o continuar
    print(f"Ocurrió un error al leer el Excel: {e}")
    print("\033[33m ⚠️Es posible que tenga el archivo abierto mientras se intenta leer, cierre el archivo para poder leerlo \033[0m")
    HOJA_SSBB_OK = False

### Leer el mensaje de arriba

Modulos para cargar las hojas 

In [108]:
# Eliminar columnas sobrantes
#columna_base = 'Glosa/Justificación'

#dfRRHH = dfRRHH.iloc[:, :dfRRHH.columns.get_loc(columna_base) + 1]

#columnas = dfRRHH.columns.tolist()
#print(columnas)
def eliminar_columnas_sobrantes(df: pd.DataFrame, columna_final: str) -> pd.DataFrame:
    columna_base = columna_final
    df = df.iloc[:, :df.columns.get_loc(columna_base) + 1]
    return df

In [109]:
# Funciones de limpieza para eliminar filas vacias y las que dicen Total
def convertir_a_numerico(df: pd.DataFrame, columna: str) -> pd.DataFrame:
    """
    Convierte la columna indicada a valores numéricos.
    Texto, vacíos o valores inválidos se convierten en NaN.
    """
    df = df.copy()
    df[columna] = pd.to_numeric(df[columna], errors="coerce")
    return df


def filtrar_mayores_igual(df: pd.DataFrame, columna: str, minimo: float) -> pd.DataFrame:
    """
    Conserva únicamente filas donde la columna >= minimo.
    Los valores NaN también se eliminan automáticamente.
    """
    df = df.copy()
    return df[df[columna] >= minimo]




def eliminar_filas_con_valor_en_columna(df: pd.DataFrame, columna: str, valor: str = "Total") -> pd.DataFrame:
    """
    Elimina las filas donde la columna indicada contiene exactamente el valor especificado.
    - df: DataFrame de entrada
    - columna: nombre de la columna a verificar
    - valor: valor que, si aparece, causa la eliminación de esa fila
    """
    df = df.copy()
    return df[df[columna] != valor]



def limpiar_por_columna(df: pd.DataFrame, columna: str, minimo: float = 1) -> pd.DataFrame:
    """
    Limpieza completa basada en una columna específica:
    - convierte la columna a numérico
    - elimina texto, vacíos y NaN
    - conserva solo valores >= minimo
    """
    df = convertir_a_numerico(df, columna)
    df = filtrar_mayores_igual(df, columna, minimo)
    df = eliminar_filas_con_valor_en_columna(df, columna="Cuenta", valor="Total")
    return df

In [110]:
# Funcion para verificar si no se rinde nada (Dataframe vacio)

def dataframe_solo_con_cabeceras(df: pd.DataFrame) -> bool:
    """
    Retorna True si el DataFrame no contiene filas de datos,
    es decir, si únicamente tiene las cabeceras.
    """
    return df.shape[0] == 0

In [111]:
#Funcion para los formatos de los RUT
def columna_normalizar_alnum(df, col, *, 
                             nombre_salida=None,
                             sep_miles=None, 
                             coma_decimal=False,
                             strip=True,
                             colapsar_decimales=True):
    """
    Procesa df[col] y devuelve df con esa columna en texto (StringDtype),
    cumpliendo: sin puntos ni caracteres especiales; solo números y letras.

    Reglas:
      1) Si el valor es numérico y es un flotante entero (p. ej., 15.0, "0015.00"):
         -> "15" (texto, solo dígitos).
      2) Si el valor es numérico con decimales reales (p. ej., 12.5):
         -> si colapsar_decimales=True: quitar el separador decimal (p. ej., "125");
            si False: conservar forma textual (normalizada), pero luego limpiar a alfanumérico (quitar punto/coma).
      3) Si el valor NO es numérico: se limpia a solo caracteres alfanuméricos (A-Za-z0-9).

    Parámetros:
      - nombre_salida: si se indica, escribe en esa columna; si no, sobrescribe `col`.
      - sep_miles: carácter separador de miles a eliminar primero (p. ej., ',', '.', ' ').
      - coma_decimal: si True, interpreta coma como separador decimal (ej.: "12,5" -> 12.5).
      - strip: recorta espacios en strings antes de procesar.
      - colapsar_decimales: True (por defecto) elimina el separador decimal en decimales reales.
                            False los deja textual (pero sin el separador al final por la limpieza alfanumérica).

    Retorna:
      - df: el mismo DataFrame, con la columna transformada a StringDtype.
    """
    destino = nombre_salida or col
    s = df[col].astype("string")

    # 1) Preparación de texto
    if strip:
        s = s.str.strip()

    # 2) Normalización opcional de formato numérico (miles/decimal)
    #    * Primero eliminamos sep_miles si se especifica.
    if sep_miles:
        s_norm = s.str.replace(sep_miles, "", regex=False)
    else:
        s_norm = s

    #    * Si coma_decimal=True, convertimos coma decimal a punto para poder parsear
    if coma_decimal:
        s_norm = s_norm.str.replace(",", ".", regex=False)

    # 3) Intento de parseo numérico
    num = pd.to_numeric(s_norm, errors="coerce")
    es_num = num.notna() & np.isfinite(num)
    es_entero_exacto = es_num & (num == np.floor(num))

    # 4) Construcción de salida como texto
    out = s.copy()

    # 4a) Flotante entero -> "entero" (sin separadores, solo dígitos)
    out.loc[es_entero_exacto] = num.loc[es_entero_exacto].astype("Int64").astype("string")

    # 4b) Numérico con decimales reales
    es_decimal_real = es_num & ~es_entero_exacto
    if es_decimal_real.any():
        if colapsar_decimales:
            # Quitamos el separador decimal unificando a texto "sin punto"
            # Ej.: 12.5 -> "125", 100.05 -> "10005"
            txt = s_norm.loc[es_decimal_real].astype("string")
            # A esta altura el decimal es con punto si coma_decimal=True se normalizó
            txt = txt.str.replace(".", "", regex=False)
            out.loc[es_decimal_real] = txt
        else:
            # Mantenemos representación textual normalizada (con '.' si correspondía),
            # luego al final limpiaremos caracteres no alfanuméricos (lo cual quitará el '.')
            out.loc[es_decimal_real] = s_norm.loc[es_decimal_real].astype("string")

    # 4c) No numéricos: se dejan para limpieza final

    # 5) Limpieza final: dejar solo alfanuméricos (A-Za-z0-9), quitar todo lo demás
    #    Si la celda es NA, se mantiene como <NA>.
    def solo_alnum(val: str):
        if val is None or pd.isna(val):
            return pd.NA
        # Eliminar todo lo que no sea [A-Za-z0-9]
        limpio = re.sub(r'[^A-Za-z0-9]', '', val)
        # Si termina vacío, lo dejamos como <NA> para no introducir cadenas vacías
        return limpio if limpio != "" else pd.NA

    out = out.map(solo_alnum)

    df[destino] = out.astype("string")
    return df

In [112]:
# Limpiar puntos flotantes en las columnas
def limpiar_decimales(df: pd.DataFrame) -> pd.DataFrame:
    """
    Elimina (trunca) la parte decimal en todo el DataFrame:
      - En columnas numéricas: devuelve enteros (pandas 'Int64' con soporte de NA).
      - En columnas no numéricas: convierte textos numéricos (p. ej. '3.14') a su parte entera como texto ('3').
      - No redondea: solo truncado hacia 0 (np.trunc).
    """
    df = df.copy()

    # 1) Columnas numéricas -> enteros (Int64), truncado
    for col in df.columns:
        if is_numeric_dtype(df[col]):
            s = pd.to_numeric(df[col], errors="coerce")
            out = np.trunc(s)  # truncado hacia 0
            out = pd.Series(out, index=s.index).replace([np.inf, -np.inf], np.nan)
            df[col] = out.astype("Int64")
    
    # 2) Columnas no numéricas -> transformar solo celdas que sean números en texto
    def trunc_str_num(x):
        # Dejar None/NaN tal cual
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return x
        # Intentar parsear a número desde string u otros tipos
        try:
            # Permitir strings con espacios o comas decimales comunes
            if isinstance(x, str):
                s = x.strip().replace(",", ".")
            else:
                s = str(x)
            val = float(s)
        except Exception:
            return x  # no parece número -> dejar igual
        # Truncar hacia 0 y devolver como string sin decimales
        if np.isinf(val) or np.isnan(val):
            return x  # conservar representaciones especiales
        return str(int(np.trunc(val)))

    for col in df.columns:
        if not is_numeric_dtype(df[col]):
            df[col] = df[col].apply(trunc_str_num)

    return df



In [113]:
#Eliminar los espacio en la glosa
def limpiar_saltos_linea(df, columna):
    # Reemplaza saltos de línea (\n y \r) por un espacio o vacío
    df[columna] = df[columna].astype(str).str.replace(r'[\r\n]+', ' ', regex=True)
    return df


In [114]:
# Modulo de procesado de la hoja 
# Unicamente verifica que la hoja existe y no se encuentra vacia
def procesar_hoja(df: pd.DataFrame, RUT1: str, RUT2: str, columna_final: str) -> pd.DataFrame:
    "Llama a todas las funciones para procesar la hoja"
    df = eliminar_columnas_sobrantes(df,columna_final) 
    df = limpiar_por_columna(df, columna="Valor Rendido al Proyecto", minimo=1) # Filas vacias o sin rendir
    df = columna_normalizar_alnum(df, RUT1) # Formato a los rut
    df = columna_normalizar_alnum(df, RUT2) # Formato a los rut
    df = limpiar_decimales(df)
    df = limpiar_saltos_linea(df, columna_final)
    if dataframe_solo_con_cabeceras(df):
        print("La Hoja no tiene nada por rendir")
        Tiene_Datos = False
    else:
        print("La Hoja tiene valores por rendir")
        Tiene_Datos = True
    return df, Tiene_Datos

Modulos para auditar

In [115]:
# Verificador de longitud de RUT 
_ZW_RE = re.compile(r"[\u200B-\u200D\uFEFF\u00A0]")

def _normalizar_str(s: str) -> str:
    s = unicodedata.normalize("NFKC", s)
    s = _ZW_RE.sub("", s)
    s = s.strip()
    return s

def imprimir_filas_invalidas_alnum9(df: pd.DataFrame, columna: str, incluir_nan: bool = True) -> None:
    if columna not in df.columns:
        raise KeyError(f"La columna '{columna}' no existe en el DataFrame.")
    patron = re.compile(r"^[A-Za-z0-9]{9}$")
    total_invalidas = 0
    for idx, row in df.iterrows():
        val = row[columna]
        if val is None or (isinstance(val, float) and math.isnan(val)):
            if incluir_nan:
                print(f"--- Fila índice {idx} (NaN/None) ---")
                print(row.to_string())
                print()
                total_invalidas += 1
            continue
        s = _normalizar_str(str(val))
        if not patron.fullmatch(s):
           
        # Datos a mostrar: 3 columnas del DataFrame
            columnas_tabla = [columna, "N° Documento" ]  # <-- remplaza por tus 3 columnas

            # Validación de columnas
            for c in columnas_tabla:
                if c not in df.columns:
                    raise KeyError(f"La columna '{c}' no existe en el DataFrame.")

            print(f"\033[33m ⚠️ Por favor verificar los RUT que no tiene 9 caracteres \033[0m")
            print(f"Valor de {columna}: {repr(s)} (Longitud: {len(s)} caracteres)")

            # Crear tabla limpia SOLO con las 3 columnas deseadas
            tabla = row[columnas_tabla].to_frame().T

            print(tabla.to_string(index=False))  # Imprime tabla ordenada
            print()


            total_invalidas += 1
    if total_invalidas == 0:
        print(f"✅ \033[32mTodas las filas de '{columna}' cumplen: exactamente 9 caracteres alfanuméricos.\033[0m")
        imprimir_CSV = True   
    else:
        print(f" Total de filas inválidas en '{columna}': {total_invalidas}")
        imprimir_CSV = False   

    return imprimir_CSV    

In [116]:
#Verificar Fechas
def comparar_fechas(df, col_x, col_y):
    # Asegurar que las columnas son tipo fecha
    df[col_x] = pd.to_datetime(df[col_x])
    df[col_y] = pd.to_datetime(df[col_y])

    # Identificar filas donde x < y
    mask = df[col_x] > df[col_y]
    Fechas_ok=True
    # Imprimir resultados
    for idx in df[mask].index:
        # Columnas que quieres mostrar en la tabla
        columnas_tabla = [col_x, col_y, "N° Documento" ]  # <- cámbialas por tus 3 columnas
        print("\n 🚫 \033[31m La fecha de emisión no puede ser mayor a la fecha de Pago \033[0m")
        # Extraer solo esas columnas y mostrarlas como tabla
        fila = df.loc[idx, columnas_tabla].to_frame().T
        print(fila.to_string(index=False))
        Fechas_ok=False
    if Fechas_ok:
        print("✅ \033[32mTodas las fechas tienen sentido.\033[0m")
    return Fechas_ok  


In [117]:
#Verificar montos totales y rendidos
def comparar_montos(df, col_x, col_y):
    # Convierte dos columnas a numérico antes de comparar
    df[col_x] = pd.to_numeric(df[col_x]
                            .astype(str)
                            .str.replace(r"[^\d\.\,\-]", "", regex=True)
                            .str.replace(".", "", regex=False)  # quita miles
                            .str.replace(",", ".", regex=False),  # coma→punto
                            errors="coerce")

    df[col_y] = pd.to_numeric(df[col_y]
                            .astype(str)
                            .str.replace(r"[^\d\.\,\-]", "", regex=True)
                            .str.replace(".", "", regex=False)
                            .str.replace(",", ".", regex=False),
                            errors="coerce")

    # Identificar filas donde x < y
    mask = df[col_x] < df[col_y]
    montos_ok=True
    # Imprimir resultados
    for idx in df[mask].index:
        print("\n 🚫 \033[31m El monto de la factura/invoice/liquidación no puede ser menor al rendido \033[0m")
        print(df.loc[idx])
        montos_ok=False
    if montos_ok:
        print("✅ \033[32mTodas los montos rendidos tienen sentido.\033[0m")
    return montos_ok  

In [118]:
#Verificar horas contratadas y rendidas
def comparar_horas(df, col_x, col_y):
    # Convierte dos columnas a numérico antes de comparar
    df[col_x] = pd.to_numeric(df[col_x]
                            .astype(str)
                            .str.replace(r"[^\d\.\,\-]", "", regex=True)
                            .str.replace(".", "", regex=False)  # quita miles
                            .str.replace(",", ".", regex=False),  # coma→punto
                            errors="coerce")

    df[col_y] = pd.to_numeric(df[col_y]
                            .astype(str)
                            .str.replace(r"[^\d\.\,\-]", "", regex=True)
                            .str.replace(".", "", regex=False)
                            .str.replace(",", ".", regex=False),
                            errors="coerce")

    # Identificar filas donde x < y
    mask = df[col_x] < df[col_y]
    horas_ok=True
    # Imprimir resultados
    for idx in df[mask].index:
        print("\n 🚫 \033[31m Las horas de la contratadas no puede ser menor a las rendidas \033[0m")
        print(df.loc[idx])
        horas_ok=False
    if horas_ok:
        print("✅ \033[32mTodas las horas tienen sentido.\033[0m")
    return horas_ok  

In [119]:
# Valores autorizados en las listas de SGP

def validar_columna(df: pd.DataFrame, Hoja: str,):
    # Listas de valores permitidos por SGP
    Forma_Pago = ["Cheque/Vale vista", "Transferencia electrónica", "Tarjeta de Crédito"]
    RRHH_Tipo_documento = ["LIQ. SUELDO","BOLETA HONORARIOS"]
    General_Tipo_documento = ["Factura","Invoice","Boleta"]
    listas_ok=True
    if Hoja == "RRHH":
        valores_permitidos=RRHH_Tipo_documento
        Valores_Cuenta=["RRHH"]

    if Hoja == "Gastos de Operación":
        valores_permitidos= General_Tipo_documento
        Valores_Cuenta=["GASTOS DIRECTOS"]    
    
    if Hoja == "Pers.Juridica":
        valores_permitidos= General_Tipo_documento
        Valores_Cuenta=["PERSONA JURÍDICA"]    
    
    if Hoja == "Arriendo":
        valores_permitidos= General_Tipo_documento
        Valores_Cuenta=["GASTOS DE ARRIENDO"]  

    if Hoja == "Servicios Básicos":
        valores_permitidos= General_Tipo_documento
        Valores_Cuenta=["SERVICIOS BÁSICOS"]  

          

     # Identificar valores no permitidos Cuenta
    valores_invalidos_Cuenta = df[~df["Cuenta"].isin(Valores_Cuenta)]["Cuenta"].unique()

    if len(valores_invalidos_Cuenta) > 0:
        listas_ok=False
        print("🚫 \033[31m  Hay valores inválidos en la columna:\033[0m", "Cuenta")
        print("➡️ Valores permitidos:", Valores_Cuenta)
        print("❌ Valores encontrados que NO son válidos:", list(valores_invalidos_Cuenta))
    else:
        print(f"✅\033[32m Todos los valores en la columna Cuenta son válidos.\033[0m")

    # Identificar valores no permitidos Tipo de documento
    valores_invalidos_Tipo_documento = df[~df["Tipo documento"].isin(valores_permitidos)]["Tipo documento"].unique()

    if len(valores_invalidos_Tipo_documento) > 0:
        listas_ok=False
        print("🚫 \033[31m  Hay valores inválidos en la columna:\033[0m", "Tipo documento")
        print("➡️ Valores permitidos:", valores_permitidos)
        print("❌ Valores encontrados que NO son válidos:", list( valores_invalidos_Tipo_documento))
    else:
        print(f"✅\033[32m Todos los valores en la columna Tipo documento son válidos.\033[0m")

    # Identificar valores no permitidos Forma de Pago
    valores_invalidos_Forma_Pago = df[~df["Forma de Pago"].isin(Forma_Pago)]["Forma de Pago"].unique()

    if len(valores_invalidos_Forma_Pago) > 0:
        listas_ok=False
        print("🚫 \033[31m  Hay valores inválidos en la columna:\033[0m", "Forma de Pago")
        print("➡️ Valores permitidos:", Forma_Pago)
        print("❌ Valores encontrados que NO son válidos:", list( valores_invalidos_Forma_Pago))
    else:
        print(f"✅\033[32m Todos los valores en la columna Forma de Pago son válidos.\033[0m")    
    return listas_ok        

In [120]:
### Modulo de Auditoria de columnas
# Unicamente verifica que la hoja existe y no se encuentra vacia
def auditar_hoja(df: pd.DataFrame, Rut1: str, Rut2: str, Monto1: str, Monto2: str, FechaEmitido: str, FechaPagado: str, Hoja: str  ) -> pd.DataFrame:
    "Llama a las funciones de verificación"
    imprimir_CSV=imprimir_filas_invalidas_alnum9(df,Rut1)
    imprimir_CSV=imprimir_filas_invalidas_alnum9(df,Rut2)  
    imprimir_CSV=comparar_fechas(df, FechaEmitido, FechaPagado)
    imprimir_CSV=comparar_montos(df, Monto1, Monto2)
    if Hoja=="RRHH":
        imprimir_CSV=comparar_horas(df, "Total de horas contratadas.", "Horas dedicadas al proyecto ")
    imprimir_CSV=validar_columna(df, Hoja)
    return imprimir_CSV

Exportar al fortamo CSV separado por ;

In [121]:
# Convertir a CSV seprado por ;
def convertir_csv(df: pd.DataFrame, Hoja: str, PI_Proyecto: str)-> pd.DataFrame:
    "Llama a las funciones de verificación"
    df.to_csv(f"{Hoja}_{PI_Proyecto}.csv", sep=';', index=False, encoding='utf-8-sig', date_format='%d/%m/%Y')
   


## <p style="color:#2898EE;"> Hoja de RRHH</p>

In [122]:
#Procesado de RRHH
columna_final="Glosa/Justificación"
Rut1="RUT Contribuyente"
Rut2="RUT RRHH"
Monto1="Total Haberes Total Boleta"
Monto2="Valor Rendido al Proyecto"
FechaEmitido="Fecha Emision Documento"
FechaPagado="Fecha de Pago Real"
Hoja="RRHH"

if not HOJA_RRHH_OK:
    print("Módulo RRHH omitido porque no existe la hoja RRHH.")
else:
    
    dfRRHH, Tiene_DatosRRHH = procesar_hoja(dfRRHH,Rut1,Rut2,columna_final)  
    
    if Tiene_DatosRRHH:
        # Convertir a CSV
        imprimir_CSV=auditar_hoja(dfRRHH, Rut1, Rut2, Monto1, Monto2, FechaEmitido, FechaPagado, Hoja)  
        if imprimir_CSV:
            convertir_csv(dfRRHH,Hoja,PI_Proyecto)
        
        else:
            print("Arreglar inconsitencia para imprimir CSV")
    else:
        print("No hay datos para revisar")



La Hoja tiene valores por rendir
 ⚠️ Por favor verificar los RUT que no tiene 9 caracteres 
Valor de RUT Contribuyente: '99576630111' (Longitud: 11 caracteres)
RUT Contribuyente N° Documento
      99576630111       122025

 Total de filas inválidas en 'RUT Contribuyente': 1
 ⚠️ Por favor verificar los RUT que no tiene 9 caracteres 
Valor de RUT RRHH: '17753263' (Longitud: 8 caracteres)
RUT RRHH N° Documento
17753263       122025

 Total de filas inválidas en 'RUT RRHH': 1
✅ Todas las fechas tienen sentido.

 🚫  El monto de la factura/invoice/liquidación no puede ser menor al rendido 
Cuenta                                               RRHH
Ítem                                             Director
Codigo                                                  1
RUT Contribuyente                               995766302
RUT RRHH                                         17753263
Nombre del RRHH                    Thomas Husak Sotomayor
Tipo documento                                LIQ. SUELDO
N° 

<p style="color:#FF7F00;">Sí, considera que el error no aplica y se puede convertir al CSV dejar True</p>

In [123]:
Ok_TODO =True

In [124]:

if Ok_TODO:
    # Convertir a CSV
    convertir_csv(dfRRHH,Hoja,PI_Proyecto)

## <p style="color:#2898EE;">  Hoja de Gastos de Operación </p>

In [125]:
#Procesado de Gastos de Operación
columna_final="Glosa / Justificación"
Rut1="RUT Contribuyente"
Rut2="RUT Proveedor"
Monto1="Monto Total Documento (V. Neto)"
Monto2="Valor Rendido al Proyecto"
FechaEmitido="Fecha Emisión Documento"
FechaPagado="Fecha Pago Real"
Hoja="Gastos de Operación"

if not HOJA_GOperacional_OK:
    print("Módulo Gastos de Operación omitido porque no existe la hoja Gastos de Operación.")
else:
    
    dfOperaciones, Tiene_DatosOperaciones = procesar_hoja(dfOperaciones,Rut1,Rut2,columna_final)  
    
    if Tiene_DatosOperaciones: 
        # Convertir a CSV
        imprimir_CSV=auditar_hoja(dfOperaciones,Rut1,Rut2,Monto1,Monto2,FechaEmitido,FechaPagado,Hoja)  
        if imprimir_CSV:
            convertir_csv(dfOperaciones,Hoja,PI_Proyecto)
        
        else:
            print("Arreglar inconsitencia para imprimir CSV")
    else:
        print("No hay datos para revisar")


La Hoja tiene valores por rendir
✅ Todas las filas de 'RUT Contribuyente' cumplen: exactamente 9 caracteres alfanuméricos.
✅ Todas las filas de 'RUT Proveedor' cumplen: exactamente 9 caracteres alfanuméricos.
✅ Todas las fechas tienen sentido.
✅ Todas los montos rendidos tienen sentido.
✅ Todos los valores en la columna Cuenta son válidos.
✅ Todos los valores en la columna Tipo documento son válidos.
✅ Todos los valores en la columna Forma de Pago son válidos.


<p style="color:#FF7F00;">Sí, considera que el error no aplica y se puede convertir al CSV dejar True</p>

In [126]:
Ok_TODO =False

In [127]:

if Ok_TODO:
    # Convertir a CSV
    convertir_csv(dfOperaciones,Hoja,PI_Proyecto)

## <p style="color:#2898EE;">  Hoja de Pers.Juridicas</p>

In [128]:
#Procesado de Pers.Juridica
columna_final="Glosa / Justificación"
Rut1="RUT Contribuyente"
Rut2="RUT Proveedor"
Monto1="Monto Total Documento (V. Neto)"
Monto2="Valor Rendido al Proyecto"
FechaEmitido="Fecha Emisión Documento"
FechaPagado="Fecha Pago Real"
Hoja="Pers.Juridica"

if not HOJA_PersJuridica_OK :
    print("Módulo Gastos de Operación omitido porque no existe la hoja Gastos de Operación.")
else:
    
    dfPersJuridica, Tiene_DatosJuridicas = procesar_hoja(dfPersJuridica,Rut1,Rut2,columna_final)  
    
    if Tiene_DatosJuridicas: 
        # Convertir a CSV
        
        imprimir_CSV=auditar_hoja(dfPersJuridica,Rut1,Rut2,Monto1,Monto2,FechaEmitido,FechaPagado,Hoja)  
        if imprimir_CSV:
            convertir_csv(dfPersJuridica,Hoja,PI_Proyecto)
        
        else:
            print("Arreglar inconsitencia para imprimir CSV")
    else:
        print("No hay datos para revisar")


La Hoja tiene valores por rendir
✅ Todas las filas de 'RUT Contribuyente' cumplen: exactamente 9 caracteres alfanuméricos.
✅ Todas las filas de 'RUT Proveedor' cumplen: exactamente 9 caracteres alfanuméricos.
✅ Todas las fechas tienen sentido.
✅ Todas los montos rendidos tienen sentido.
✅ Todos los valores en la columna Cuenta son válidos.
✅ Todos los valores en la columna Tipo documento son válidos.
✅ Todos los valores en la columna Forma de Pago son válidos.


<p style="color:#FF7F00;">Sí, considera que el error no aplica y se puede convertir al CSV dejar True</p>

In [129]:
Ok_TODO =False

In [130]:

if Ok_TODO:
    # Convertir a CSV
    convertir_csv(dfPersJuridica,Hoja,PI_Proyecto)

## <p style="color:#2898EE;">  Hoja de Arriendo </p>

In [131]:
#Procesado de Arriendo
columna_final="Glosa / Justificación"
Rut1="RUT Contribuyente"
Rut2="RUT Proveedor"
Monto1="Monto Total Documento (V. Neto)"
Monto2="Valor Rendido al Proyecto"
FechaEmitido="Fecha Emisión Documento"
FechaPagado="Fecha Pago Real"
Hoja="Arriendo"

if not HOJA_Arriendo_OK :
    print("Módulo Gastos de Operación omitido porque no existe la hoja Gastos de Operación.")
else:
    
    dfArriendo, Tiene_DatosArriendo = procesar_hoja(dfArriendo,Rut1,Rut2,columna_final)  
    
    if Tiene_DatosArriendo: 
        # Convertir a CSV
        imprimir_CSV=auditar_hoja(dfArriendo,Rut1,Rut2,Monto1,Monto2,FechaEmitido,FechaPagado,Hoja)  
        if imprimir_CSV:
            convertir_csv(dfArriendo,Hoja,PI_Proyecto)
        
        else:
            print("Arreglar inconsitencia para imprimir CSV")
    else:
        print("No hay datos para revisar")

La Hoja tiene valores por rendir
✅ Todas las filas de 'RUT Contribuyente' cumplen: exactamente 9 caracteres alfanuméricos.
✅ Todas las filas de 'RUT Proveedor' cumplen: exactamente 9 caracteres alfanuméricos.
✅ Todas las fechas tienen sentido.
✅ Todas los montos rendidos tienen sentido.
🚫   Hay valores inválidos en la columna: Cuenta
➡️ Valores permitidos: ['GASTOS DE ARRIENDO']
❌ Valores encontrados que NO son válidos: ['ARRIENDO']
✅ Todos los valores en la columna Tipo documento son válidos.
✅ Todos los valores en la columna Forma de Pago son válidos.
Arreglar inconsitencia para imprimir CSV


<p style="color:#FF7F00;">Sí, considera que el error no aplica y se puede convertir al CSV dejar True</p>

In [132]:
Ok_TODO =False

In [133]:

if Ok_TODO:
    # Convertir a CSV
    convertir_csv(dfArriendo,Hoja,PI_Proyecto)

## <p style="color:#2898EE;">  Hoja de Servicios Básicos </p>

In [134]:
#Procesado de Servicios Básicos
columna_final="Glosa / Justificación"
Rut1="RUT Contribuyente"
Rut2="RUT Proveedor"
Monto1="Monto Total Documento (V. Neto)"
Monto2="Valor Rendido al Proyecto"
FechaEmitido="Fecha Emisión Documento"
FechaPagado="Fecha Pago Real"
Hoja="Servicios Básicos"

if not HOJA_SSBB_OK :
    print("Módulo Gastos de Operación omitido porque no existe la hoja Gastos de Operación.")
else:
    
    dfSSBB, Tiene_DatosSSBB = procesar_hoja(dfSSBB,Rut1,Rut2,columna_final)  
    
    if Tiene_DatosSSBB: 
        # Convertir a CSV
        imprimir_CSV=auditar_hoja(dfSSBB,Rut1,Rut2,Monto1,Monto2,FechaEmitido,FechaPagado,Hoja)  
        if imprimir_CSV:
            convertir_csv(dfSSBB,Hoja,PI_Proyecto)
        
        else:
            print("Arreglar inconsitencia para imprimir CSV")
    else:
        print("No hay datos para revisar")

La Hoja no tiene nada por rendir
No hay datos para revisar


<p style="color:#FF7F00;">Sí, considera que el error no aplica y se puede convertir al CSV dejar True</p>

In [135]:
Ok_TODO =False

In [136]:

if Ok_TODO:
    # Convertir a CSV
    convertir_csv(dfSSBB,Hoja,PI_Proyecto)